In [1]:
uuid_alumno = "ff3f09cc-529b-4011-a745-256ac2565010"

# Sistema RAG Conversacional sobre Regulación e Inteligencia Artificial Agéntica (Versión 4 - SOTA)

**Autor:** Alumno de Máster en Inteligencia Artificial y Ciencia de Datos (`ff3f09cc-529b-4011-a745-256ac2565010`)  
**Asignatura:** Inteligencia Artificial Agéntica y Procesamiento del Lenguaje Natural  
**Objetivo Práctico (Arquitectura V4):** Construir y auditar un motor conversacional modular de consulta legal sobre Inteligencia Artificial y Privacidad (`RGPD`, `Guía de IA Agéntica` y `Dictamen del EDPB`) utilizando **LangGraph por Nodos No Lineales (Enrutamiento Condicional ReAct)**, **Recuperación Híbrida MMR en Dos Etapas (`ChromaDB top_k=4`)**, **Aislamiento de Memoria por Hilo (`MemorySaver + thread_id`)** y segmentación jerárquica sobre documentos en formato Markdown (`.md`).

---

## Preparación del Entorno y Control de Errores Inicial

En este primer bloque importamos las herramientas técnicas del ecosistema Python necesarias para construir la arquitectura V4:
* **LangChain y LangGraph:** Para cargar documentos, trocear texto estructurado, instanciar el checkpointer de memoria (`MemorySaver`) y construir el grafo condicional bifurcado con enrutador.
* **ChromaDB:** Para gestionar nuestra base de datos vectorial en disco local con búsqueda de relevancia marginal máxima (`MMR`).
* **Google GenAI / Gemini:** Para vectorizar los textos (`embeddings`) y realizar la inferencia conversacional con `gemini-3.1-flash-lite`.
* **IPython.display:** Para renderizar visualmente las respuestas con formato Markdown jerárquico dentro de las celdas del cuaderno.

Implementamos una comprobación de seguridad temprana para verificar que la variable de entorno `GEMINI_API_KEY` esté cargada y disponible en el entorno (.env).


In [2]:
# Importacion de librerias requeridas para el motor RAG conversacional
import os
import time
import re
from typing import List, Dict, Any
from typing_extensions import TypedDict

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
import uuid

# Herramientas de renderizado visual en Jupyter Notebook
from IPython.display import display, Markdown, HTML

# Carga automática de .env si existe en el directorio raíz
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Verificacion de la clave de acceso a la API de Gemini
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("[ERROR CRITICO]: No se ha detectado la clave GEMINI_API_KEY en las variables de entorno. Por favor, configurala antes de ejecutar el cuaderno.")
else:
    print("[INFO] Clave GEMINI_API_KEY detectada correctamente en el entorno de ejecucion.")


C:\Users\andre\AppData\Local\Temp\ipykernel_30576\1689121126.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[INFO] Clave GEMINI_API_KEY detectada correctamente en el entorno de ejecucion.


---

## Carga del Corpus Normativo (Rutas Relativas para Portabilidad)

Cargamos los tres ficheros normativos en formato Markdown desde nuestra subcarpeta `data/raw/`:
* `EDPB_Opinion_2024_28.md` (Dictamen del Comité Europeo de Protección de Datos en inglés sobre modelos de IA).
* `Orientaciones Ia Agéntica.md` (Guía práctica en español sobre agentes autónomos y privacidad).
* `reglamentoRGPD.md` (Texto estructurado del Reglamento General de Protección de Datos).

**Regla de Portabilidad del Proyecto:** Queda estrictamente prohibido utilizar rutas absolutas locales (como `C:\Users\...` o `/home/alumno/...`). Como el cuaderno de Jupyter se ejecuta desde la carpeta `notebooks/`, utilizamos `os.path.join("..", "data", "raw")` para retroceder un directorio y llegar a la raíz de datos. De esta forma, el proyecto es totalmente portable y funcionará a la primera en el ordenador del evaluador sin necesidad de modificar rutas.

Para no sobrecargar la vista del cuaderno con cientos de líneas de texto plano, imprimimos únicamente una tabla resumen que verifica la correcta lectura de cada archivo y muestra su peso en KB y cantidad de caracteres.


In [3]:
# Carga limpia y estructurada utilizando rutas relativas
# Al estar el cuaderno situado en la subcarpeta /notebooks, retrocedemos un nivel con '..'
ruta_raw = os.path.join("..", "data", "raw")

archivos_md = [
    "EDPB_Opinion_2024_28.md",
    "Orientaciones Ia Agéntica.md",
    "reglamentoRGPD.md"
]

documentos_crudos = []
print("[INFO] Cargando ficheros normativos desde la carpeta data/raw/:")
print(f"{'Archivo normativo':<35} | {'Tamano (KB)':<12} | {'Caracteres':<12}")
print("-" * 65)

for archivo in archivos_md:
    ruta_completa = os.path.join(ruta_raw, archivo)
    if not os.path.exists(ruta_completa):
        raise FileNotFoundError(f"[ERROR]: No se encuentra el archivo {archivo} en {ruta_raw}. Verifica la estructura de carpetas.")
    
    loader = TextLoader(ruta_completa, encoding="utf-8")
    docs = loader.load()
    documentos_crudos.extend(docs)
    
    tamano_kb = os.path.getsize(ruta_completa) / 1024
    num_caracteres = len(docs[0].page_content)
    print(f"{archivo:<35} | {tamano_kb:<12.2f} | {num_caracteres:<12}")

print("-" * 65)
print(f"[INFO] Carga completada con exito. Documentos procesados: {len(documentos_crudos)}")


[INFO] Cargando ficheros normativos desde la carpeta data/raw/:
Archivo normativo                   | Tamano (KB)  | Caracteres  
-----------------------------------------------------------------
EDPB_Opinion_2024_28.md             | 93.83        | 96079       
Orientaciones Ia Agéntica.md        | 41.74        | 42085       
reglamentoRGPD.md                   | 388.82       | 388886      
-----------------------------------------------------------------
[INFO] Carga completada con exito. Documentos procesados: 3


---

## Troceado Semántico y Jerárquico en Dos Etapas (Two-Stage Chunking)

Para transformar las leyes completas en fragmentos útiles para el buscador vectorial sin perder la estructura de la norma, aplicamos un procedimiento de segmentación en dos etapas:

* **Etapa 1 (`MarkdownHeaderTextSplitter`):** Le indicamos al cortador que reconozca tres niveles jerárquicos de cabeceras Markdown:
  * `#` para extraer y etiquetar el `Titulo_Ley`
  * `##` para extraer y etiquetar el `Capitulo` o sección principal
  * `###` para extraer y etiquetar el `Articulo` o subsección
  Al procesar los textos, este divisor corta el documento por las fronteras exactas de las secciones y guarda esas etiquetas dentro de los metadatos de cada bloque. Mantenemos el parámetro `strip_headers=False` para que el título también permanezca escrito dentro del propio texto del trozo, reforzando la señal temática para el vector de embedding.

* **Etapa 2 (`RecursiveCharacterTextSplitter`):** En textos normativos reales existen artículos muy breves y otros extensísimos (como los artículos de multas o sanciones del RGPD). Si mandamos un bloque gigante a vectorizar, el modelo pierde precisión semántica al promediar ideas diversas. Por ello, pasamos los bloques resultantes de la Etapa 1 por un segundo divisor configurado a 1.000 caracteres como máximo, con un margen de solapamiento (`overlap`) de 150 caracteres para que las frases y párrafos mantengan continuidad estructural. La ventaja clave de este diseño es que la segunda etapa conserva intacta toda la genealogía jerárquica de metadatos obtenida en la primera.

Al ejecutar la celda mostramos el conteo total de fragmentos resultantes y una muestra técnica de los metadatos generados para verificar su correcta estructuración.


In [4]:
# Etapa 1: Corte estructural respetando los encabezados legales
headers_to_split_on = [
    ("#", "Titulo_Ley"),
    ("##", "Capitulo"),
    ("###", "Articulo"),
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False  # Conservamos la cabecera dentro del texto para enriquecer la busqueda semantica
)

chunks_estructurales = []
for doc in documentos_crudos:
    splits = md_splitter.split_text(doc.page_content)
    for s in splits:
        # Aseguramos un nombre de archivo limpio en los metadatos
        s.metadata["source"] = os.path.basename(doc.metadata.get("source", "desconocido"))
    chunks_estructurales.extend(splits)

# Etapa 2: Corte por tamano maximo con doble barra invertida en saltos de linea para evitar errores de sintaxis al exportar
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_finales = char_splitter.split_documents(chunks_estructurales)

print("[INFO] Resumen de la segmentacion semantica:")
print(f"       - Bloques tras corte estructural por cabeceras (Etapa 1): {len(chunks_estructurales)}")
print(f"       - Trozos finales listos para indexar (Etapa 2): {len(chunks_finales)}")
print("-" * 65)
print("[INFO] Ejemplo de metadatos jerarquicos generados en un trozo de muestra:")
muestra_idx = min(50, len(chunks_finales) - 1)
for clave, valor in chunks_finales[muestra_idx].metadata.items():
    print(f"       * {clave}: {valor}")


[INFO] Resumen de la segmentacion semantica:
       - Bloques tras corte estructural por cabeceras (Etapa 1): 293
       - Trozos finales listos para indexar (Etapa 2): 903
-----------------------------------------------------------------
[INFO] Ejemplo de metadatos jerarquicos generados en un trozo de muestra:
       * Titulo_Ley: **Opinion 28/2024 on certain data protection aspects related to the processing of personal data in the context of AI models**
       * Capitulo: **3 On the merits of the request**
       * Articulo: **3.1 On the nature of AI models in relation to the definition of personal data**
       * source: EDPB_Opinion_2024_28.md


---

## Base de Datos Vectorial Local y Control de Cuotas de API

En este bloque creamos la base de datos vectorial donde residirá todo el conocimiento normativo procesado. Para ello utilizamos **ChromaDB** guardado localmente en disco, y el modelo de vectorización `models/gemini-embedding-001` de Google AI Studio.

Durante las pruebas prácticas de indexación nos encontramos con un desafío técnico habitual al operar en cuentas gratuitas en la nube: si enviamos los casi 300 trozos de texto a vectorizar todos de golpe, la API rechaza las peticiones por superar el límite máximo de peticiones por minuto (error 429 por saturación o cuota de rate-limiting). Para solucionarlo y garantizar un script estable y tolerante a fallos, implementamos tres defensas técnicas:

* **Ruta relativa de persistencia en disco (`../vector_db`):** Toda la base vectorial se persiste en la carpeta local `vector_db` situada en el directorio raíz del proyecto.
* **Comprobación condicional inteligente:** Antes de iniciar llamadas de red, el script consulta si la colección `corpus_normativo_v3` ya existe en disco y tiene vectores almacenados. Si ya existe y la variable de control `forzar_reindexacion` es `False`, se omite por completo el bucle de embedding. Así cargamos la colección en milisegundos sin consumir peticiones de la cuota diaria.
* **Bucle de indexación por lotes reducidos con resiliencia de tolerancia a fallos (`Backoff reactivo`):** Cuando se indexa desde cero o cuando se fuerza la reindexación (`forzar_reindexacion = True`), el sistema agrupa los fragmentos en lotes de 15 trozos intercalando una pausa ligera de 2 segundos entre lotes. Al operar sobre la capa gratuita de Google AI Studio (`Free Tier`, con límite estricto de 100 peticiones por minuto), es esperable que durante la vectorización masiva de los 293 fragmentos se alcancen picos temporales de saturación (`Error 429 RESOURCE_EXHAUSTED`). Nuestro bloque defensivo intercepta estos eventos, informa por pantalla e inyecta exactamente 65 segundos de pausa para reiniciar el reloj de cuota móvil de Google antes de reintentar el lote (mostrando el progreso `Reintento 1/3 para este lote concreto`). Gracias a esta arquitectura de resiliencia y tolerancia a fallos, el sistema se recupera de forma autónoma y finaliza la indexación al 100% (293 vectores persistidos) sin que el kernel de Python colapse ni se pierda un solo dato.


In [5]:
# Configuracion del modelo de embeddings de Gemini
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

# Definicion de la ruta relativa local en ../vector_db
ruta_chroma = os.path.join("..", "vector_db")
vectorstore = Chroma(
    collection_name="corpus_normativo_v3",
    embedding_function=embeddings,
    persist_directory=ruta_chroma
)

# Control condicional de persistencia para proteger la cuota de la API
forzar_reindexacion = False  # Cambia a True si editas algun archivo en data/raw/ y necesitas regenerar los vectores
vectores_existentes = vectorstore._collection.count()

if vectores_existentes > 0 and not forzar_reindexacion:
    print(f"[EXITO] Base de datos vectorial detectada en disco ({ruta_chroma}).")
    print(f"[INFO] Se han cargado directamente {vectores_existentes} vectores pre-indexados en memoria.")
    print("[INFO] Se omite el bucle de embedding para ahorrar tiempo y cuota diaria de peticiones de la API.")
else:
    if forzar_reindexacion and vectores_existentes > 0:
        print("[INFO] forzar_reindexacion=True: Limpiando coleccion anterior en ChromaDB...")
        vectorstore.delete_collection()
        vectorstore = Chroma(
            collection_name="corpus_normativo_v3",
            embedding_function=embeddings,
            persist_directory=ruta_chroma
        )
    
    print("[INFO] Iniciando indexacion vectorial en ChromaDB mediante lotes controlados...")
    t_inicio = time.time()
    
    batch_size = 15  # Lotes reducidos de 15 trozos para no exceder los tokens por minuto
    for i in range(0, len(chunks_finales), batch_size):
        lote = chunks_finales[i : i + batch_size]
        # El contador de reintentos se reinicia intencionalmente a 0 para cada lote
        # independiente, permitiendo hasta 3 intentos por cada bloque de 15 trozos.
        reintentos = 0
        max_reintentos = 3
        
        while reintentos <= max_reintentos:
            try:
                vectorstore.add_documents(lote)
                break
            except Exception as e:
                reintentos += 1
                if reintentos > max_reintentos:
                    print(f"[ERROR CRITICO] Fallo la indexacion del lote {i} tras {max_reintentos} reintentos: {e}")
                    raise e
                
                # Pausa prudencial de 65s ante saturacion 429 para permitir el reinicio de la cuota del servidor
                tiempo_espera = 65.0
                print(f"[ADVERTENCIA] Error de cuota o saturacion de red ({e}). Esperando {tiempo_espera}s para reiniciar el contador de Google (Reintento {reintentos}/{max_reintentos} para este lote concreto)...")
                time.sleep(tiempo_espera)
        
        # Pausa ligera entre lotes en condiciones normales
        time.sleep(2.0)
    
    t_total = time.time() - t_inicio
    print(f"[EXITO] Indexacion completada. {len(chunks_finales)} vectores persistidos en {t_total:.2f} segundos.")


[EXITO] Base de datos vectorial detectada en disco (..\vector_db).
[INFO] Se han cargado directamente 903 vectores pre-indexados en memoria.
[INFO] Se omite el bucle de embedding para ahorrar tiempo y cuota diaria de peticiones de la API.


---

## Verificación Aislada del Motor de Búsqueda Vectorial (Paso 2 del Enunciado)

Antes de acoplar el índice vectorial al grafo de LangGraph o conectar el modelo de lenguaje (LLM), el Paso 2 de las especificaciones exige **verificar de manera independiente que la colección vectorial en ChromaDB responde correctamente a consultas de prueba en crudo (`similarity_search`)**.

Esta prueba confirma dos aspectos críticos del sistema de recuperación:
1. **Integridad del Índice:** Que el buscador recupera los fragmentos normativos adecuados utilizando la distancia del coseno sin depender de la inferencia del LLM.
2. **Preservación Jerárquica:** Que cada documento recuperado conserva intactos los metadatos de su árbol normativo (`source`, `Titulo_Ley`, `Capitulo`, `Articulo`) para alimentar con precisión la trazabilidad jurídica posterior.


In [6]:
# Verificacion aislada del retriever en crudo (Sin intervencion del LLM)
consulta_test = "¿En qué condiciones es lícito el tratamiento de datos personales según el RGPD y qué papel juega el consentimiento?"
print(f"[PASO 2 - AUDITORIA DE RECUPERACION] Ejecutando busqueda vectorial de prueba en crudo (top_k=2)...")
print(f"Consulta de control: '{consulta_test}'\n" + "="*75)

resultados_test = vectorstore.similarity_search(consulta_test, k=2)

if resultados_test:
    for idx, doc in enumerate(resultados_test, 1):
        fuente = doc.metadata.get("source", "desconocido")
        titulo = doc.metadata.get("Titulo_Ley", "")
        capitulo = doc.metadata.get("Capitulo", "")
        articulo = doc.metadata.get("Articulo", "")
        jerarquia = f"{titulo} -> {capitulo} -> {articulo}".strip(" ->")
        
        print(f"\n[RESULTADO #{idx}] Fuente: {fuente} | Estructura: [{jerarquia}]")
        print(f"Extracto del fragmento recuperado (primeros 250 caracteres):")
        print("-" * 75)
        print(doc.page_content[:250].replace("\n", " ") + "...")
        print("-" * 75)
    print("\n[EXITO VERIFICADO] El indice vectorial responde correctamente y conserva los metadatos jerarquicos.")
else:
    print("[ERROR] No se recuperaron documentos en la prueba aislada del retriever.")


[PASO 2 - AUDITORIA DE RECUPERACION] Ejecutando busqueda vectorial de prueba en crudo (top_k=2)...
Consulta de control: '¿En qué condiciones es lícito el tratamiento de datos personales según el RGPD y qué papel juega el consentimiento?'

[RESULTADO #1] Fuente: reglamentoRGPD.md | Estructura: [REGLAMENTOS -> **de 27 de abril de 2016**]
Extracto del fragmento recuperado (primeros 250 caracteres):
---------------------------------------------------------------------------
- (40) Para que el tratamiento sea lícito, los datos personales deben ser tratados con el consentimiento del interesado o sobre alguna otra base legítima establecida conforme a Derecho, ya sea en el presente Reglamento o en virtud de   <u>ES</u>   ot...
---------------------------------------------------------------------------

[RESULTADO #2] Fuente: reglamentoRGPD.md | Estructura: [REGLAMENTOS -> **de 27 de abril de 2016**]
Extracto del fragmento recuperado (primeros 250 caracteres):
----------------------------------

---

## Orquestación Conversacional No Lineal con LangGraph V4 (Enrutador + MMR + MemorySaver)

Para superar las limitaciones del clásico flujo lineal (`retrieve -> generate`) y resolver el **Dilema de Precisión vs. Cobertura en consultas generales del dominio**, construimos un grafo condicional bifurcado con **LangGraph V4** dotado de las siguientes innovaciones arquitectónicas:

### 1. Enrutamiento Condicional (`router_node` y Aristas Condicionales)
Inyectamos como primer nodo del grafo un clasificador de intención que evalúa la última pregunta con `gemini-3.1-flash-lite` (`temperatura=0.0`) y bifurca la ejecución en tres rutas exclusivas:
* **`LEGAL_RAG`:** Consultas normativas (RGPD, DPO, multas, IA Agéntica). Deriva a reformulación (`rewrite_query`) y búsqueda vectorial (`retrieve`).
* **`GENERAL_LLM`:** Saludos (*Hola*, *Buenos días*) o preguntas informáticas de cortesía/definición universal. Deriva a `direct_llm_node`, respondiendo de forma fluida sin gastar cuota ni tiempo en ChromaDB.
* **`OUT_OF_DOMAIN`:** Deportes, recetas o entretenimiento. Deriva a `out_of_domain_node`, interceptando al instante en 0.1s (`Guardrail #3`).

### 2. Recuperación en Dos Etapas (`Maximal Marginal Relevance - MMR top_k=4`)
En el nodo `retrieve_node`, sustituimos el coseno simple por **MMR (`k=4, fetch_k=20, lambda_mult=0.7`)**. El sistema recupera primero los 20 candidatos más cercanos y selecciona los 4 que maximizan relevancia y diversidad estructural, otorgando al LLM una visión panorámica ante preguntas conceptuales (*¿Cuáles son los principios del RGPD?*).

### 3. Aislamiento de Memoria por Hilo (`MemorySaver + thread_id`)
Instanciamos `checkpointer = MemorySaver()`, garantizando que cada caso o sesión de auditoría tenga un `thread_id` estanco sin contaminar conversaciones pasadas.


In [7]:
# Importación explícita para evitar NameError en entornos limpios
from langchain_core.messages import BaseMessage

# Definicion del estado para el grafo V4 en LangGraph
class GraphState(TypedDict, total=False):
    messages: List[BaseMessage]
    query_reformulada: str
    documentos_recuperados: List[Any]
    intencion: str

# Instanciacion del modelo LLM conversacional agil para V4
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.2,
    max_tokens=1500,
    max_retries=6,
    google_api_key=api_key
)

# 1. Nodo Enrutador de Intencion (Clasificador de bajo coste)
def router_node(state: GraphState) -> Dict[str, Any]:
    messages = state["messages"]
    ultima_pregunta = messages[-1].content if messages else ""
    prompt_router = f"""Analiza la pregunta del usuario y clasificala exactamente en UNA de las tres opciones siguientes (devuelve UNICAMENTE la palabra clave):
- LEGAL_RAG: Si la consulta versa sobre derecho, normativa, RGPD, articulos, multas, DPO, auditoria de inteligencia artificial, modelos de IA o privacidad europea.
- GENERAL_LLM: Si es un saludo conversacional (ej. 'Hola', 'Buenos dias', 'Que tal') o una pregunta abstracta/informatica de cortesía sin relacion especifica con legislacion europea ni documentos legales concretos.
- OUT_OF_DOMAIN: Si la consulta trata sobre deportes, futbol, recetas de cocina, entretenimiento, o temas ajenos a tecnologia, IA y derecho.

Pregunta: "{ultima_pregunta}"
Clasificacion:"""
    resp = llm.invoke([HumanMessage(content=prompt_router)])
    cat_raw = resp.content
    if isinstance(cat_raw, list):
        cat_raw = "".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in cat_raw])
    cat = str(cat_raw).strip()
    if "OUT_OF_DOMAIN" in cat:
        return {"intencion": "OUT_OF_DOMAIN"}
    elif "GENERAL_LLM" in cat:
        return {"intencion": "GENERAL_LLM"}
    else:
        return {"intencion": "LEGAL_RAG"}

# 2. Nodos de respuesta directa (Saludos y Fuera de Dominio)
def direct_llm_node(state: GraphState) -> Dict[str, Any]:
    messages = state["messages"]
    ultima_pregunta = messages[-1].content if messages else ""
    prompt_direct = f"""Eres un Consultor Legal y Tecnológico de Alto Nivel. El usuario te ha dirigido un saludo o consulta conversacional: "{ultima_pregunta}".
Responde de forma profesional, educada y breve en el exacto mismo idioma que el usuario, indicando tu especialidad y disposición para auditar normativas de IA y RGPD."""
    resp = llm.invoke([HumanMessage(content=prompt_direct)])
    contenido = resp.content
    if isinstance(contenido, list):
        contenido = "".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in contenido])
    contenido = str(contenido).strip()
    return {"messages": messages + [AIMessage(content=contenido)]}

def out_of_domain_node(state: GraphState) -> Dict[str, Any]:
    messages = state["messages"]
    return {"messages": messages + [AIMessage(content="No estoy entrenado para responder sobre ese tema")]}

# 3. Nodo Reformulador de Anáforas (Para ruta RAG)
def rewrite_query_node(state: GraphState) -> Dict[str, Any]:
    messages = state["messages"]
    ultima_pregunta = messages[-1].content if messages else ""
    if len(messages) <= 1:
        return {"query_reformulada": ultima_pregunta}
    prompt_reformulacion = f"""Historial reciente del chat:
{[m.content for m in messages[-4:-1]]}

Pregunta actual del usuario: "{ultima_pregunta}"

Tu tarea tecnica es reformular la pregunta actual para que sea independiente y autocontenida, reemplazando pronombres o referencias implicitas por los conceptos de los turnos anteriores. Devuelve unicamente la pregunta reformulada sin preambulos ni explicaciones."""
    respuesta = llm.invoke([HumanMessage(content=prompt_reformulacion)])
    contenido_req = respuesta.content
    if isinstance(contenido_req, list):
        contenido_req = "".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in contenido_req])
    return {"query_reformulada": str(contenido_req).strip()}

# 4. Nodo de Recuperación Híbrida MMR en Dos Etapas (top_k=4 con máxima diversidad)
def retrieve_node(state: GraphState) -> Dict[str, Any]:
    query = state.get("query_reformulada") or state["messages"][-1].content
    try:
        retriever = vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.7}
        )
        docs = retriever.invoke(query)
        return {"documentos_recuperados": docs}
    except Exception as e:
        print(f"[ADVERTENCIA] Error en la consulta vectorial MMR: {e}")
        return {"documentos_recuperados": []}

# 5. Nodo de Generación Legal RAG (Con Guardrails e Instrucción Elástica de Síntesis)
template_rag = """[PERFIL Y TONO]
Eres un Consultor Legal Senior y Arquitecto de Cumplimiento Normativo en Inteligencia Artificial y Privacidad (RGPD / EDPB / IA Agéntica).
Tu lenguaje debe ser formal, preciso, analítico, riguroso y estrictamente acotado al conocimiento normativo facilitado.

[REGLAS DE COMPORTAMIENTO ESTRATEGICO Y GUARDRAILS]
1. Responde UNICAMENTE utilizando la información técnica y normativa presente en los fragmentos de [CONTEXTO DE OPERACION RECUPERADO]. No inventes ni supongas nada fuera de estos textos.
2. Si el usuario hace referencia a una pregunta o respuesta anterior, consulta el [HISTORIAL DE CONVERSACION RECIENTE] para mantener coherencia absoluta.
3. GUARDRAIL DE DOMINIO: Si la pregunta del usuario NO está relacionada en absoluto con el ámbito legal, tecnológico, inteligencia artificial o protección de datos (ej. recetas de cocina, deportes, literatura general), debes responder exactamente: "No estoy entrenado para responder sobre ese tema".
4. GUARDRAIL DE AUSENCIA DE INFORMACION: Si la pregunta está dentro del dominio jurídico/tecnológico, pero la respuesta no se encuentra en los fragmentos recuperados, tu única respuesta debe ser: "La información disponible en la base de conocimientos no permite responder a esta consulta".
5. GUARDRAIL DE IDIOMA (ESPEJO LINGÜÍSTICO OBLIGATORIO): Detecta el idioma en que está escrita la "Pregunta actual del usuario" y redacta tu dictamen SIEMPRE en ese exacto mismo idioma:
   - Si la "Pregunta actual del usuario" está en ESPAÑOL, el 100% de tu respuesta (Conclusión Directa, Análisis Normativo y Trazabilidad Jurídica) DEBE estar estrictamente en ESPAÑOL. Queda terminantemente prohibido usar inglés.
   - Si la "Pregunta actual del usuario" está en INGLÉS, el 100% de tu dictamen DEBE redactarse estrictamente en INGLÉS, traduciendo al inglés cualquier concepto legal de los textos en español.

[FORMATO DE SALIDA OBLIGATORIO Y ESPEJO LINGÜÍSTICO]
Detecta el idioma en que está escrita la "Pregunta actual del usuario". El 100% de tu texto, incluyendo estrictamente los encabezados y el contenido de cada sección, DEBE estar exactamente en el mismo idioma de la pregunta:

- SI LA PREGUNTA ESTÁ EN ESPAÑOL:
  Estructura tu respuesta utilizando estos tres encabezados exactos e inalterados SIN NUMERAR y redacta el contenido en español:
  ### Conclusión Directa
  Una única frase corta, profesional y categórica respondiendo de forma clara e inequívoca a la consulta planteada.
  ### Análisis Normativo
  Explicación técnica detallada y fundamentada en viñetas ordenadas o párrafos breves, basada exclusivamente en los preceptos y considerandos de los fragmentos recuperados.
  ### Trazabilidad Jurídica
  Indica al final de tu respuesta las fuentes exactas de donde extrajiste cada afirmación. REGLA DE DESDUPLICACION OBLIGATORIA: Si varios fragmentos utilizados pertenecen exactamente a la misma ley, capítulo y artículo o sección, agrupa las referencias y escribe esa cabecera UNA SOLA VEZ en la lista final. Está estrictamente prohibido repetir citas legales idénticas. Utiliza viñetas tipográficas limpias con el formato:
  * **<Nombre_Archivo> (<Titulo_Ley> -> <Capitulo> -> <Articulo>)** — Breve mención al precepto.

- SI LA PREGUNTA ESTÁ EN INGLÉS:
  Estructura tu respuesta utilizando estrictamente estos tres encabezados exactos en inglés SIN NUMERAR y TRADUCE COMPLETAMENTE AL INGLÉS todo el dictamen (Conclusión, Análisis Normativo y justificación de las fuentes):
  ### Direct Conclusion
  A single short, professional, and definitive sentence clearly and unequivocally answering the query.
  ### Normative Analysis
  Detailed technical explanation in ordered bullet points or short paragraphs, based exclusively on the provisions and recitals of the retrieved fragments. You must translate all Spanish legal concepts to precise English terminology.
  ### Legal Traceability
  List at the end the exact sources from which each statement was extracted. MANDATORY DEDUPLICATION RULE: If multiple fragments belong to the exact same law, chapter, and article or section, group the references and write that header ONLY ONCE in the final list. Use clean typographic bullets with the format:
  * **<File_Name> (<Law_Title> -> <Chapter> -> <Article>)** — Brief mention of the legal provision translated to English.

[HISTORIAL DE CONVERSACION RECIENTE]
{chat_history}

[CONTEXTO DE OPERACION RECUPERADO]
{context}

Pregunta actual del usuario: 
{question}
"""

def generate_node(state: GraphState) -> Dict[str, Any]:
    messages = state["messages"]
    docs = state.get("documentos_recuperados", [])
    query = state.get("query_reformulada") or messages[-1].content
    if not docs:
        msg_fallo = "La información disponible en la base de conocimientos no permite responder a esta consulta."
        return {"messages": messages + [AIMessage(content=msg_fallo)]}
        
    contexto_estratificado = []
    for d in docs:
        meta = d.metadata
        src = meta.get("source", "Documento Desconocido")
        titulo_ley = meta.get("titulo_ley", "General")
        cap = meta.get("capitulo", "Sin Capitulo")
        art = meta.get("articulo_o_seccion", "Sin Articulo")
        jerarquia = f"{titulo_ley} -> {cap} -> {art}"
        contexto_estratificado.append(f"--- DOCUMENTO: {src} | ESTRUCTURA: [{jerarquia}] ---\n{d.page_content}")
        
    texto_contexto = "\n\n".join(contexto_estratificado)
    historial_str = ""
    if len(messages) > 1:
        turnos_recientes = messages[-5:-1] if len(messages) > 5 else messages[:-1]
        for m in turnos_recientes:
            rol = "Usuario" if isinstance(m, HumanMessage) else "Consultor"
            historial_str += f"{rol}: {m.content}\n"
            
    prompt_evaluado = template_rag.format(
        chat_history=historial_str or "Sin historial previo.",
        context=texto_contexto,
        question=query
    )
    respuesta = llm.invoke([HumanMessage(content=prompt_evaluado)])
    contenido = respuesta.content
    if isinstance(contenido, list):
        contenido = "".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in contenido])
    contenido = str(contenido).strip()
    return {"messages": messages + [AIMessage(content=contenido)]}

# Ensamblaje del Grafo V4 (Router Condicional + MMR + Checkpointer MemorySaver)
workflow = StateGraph(GraphState)
workflow.add_node("router", router_node)
workflow.add_node("rewrite_query", rewrite_query_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)
workflow.add_node("direct_llm", direct_llm_node)
workflow.add_node("out_of_domain", out_of_domain_node)

def route_decision(state: GraphState) -> str:
    return state.get("intencion", "LEGAL_RAG")

workflow.add_edge(START, "router")
workflow.add_conditional_edges("router", route_decision, {
    "LEGAL_RAG": "rewrite_query",
    "GENERAL_LLM": "direct_llm",
    "OUT_OF_DOMAIN": "out_of_domain"
})
workflow.add_edge("rewrite_query", "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)
workflow.add_edge("direct_llm", END)
workflow.add_edge("out_of_domain", END)

checkpointer = MemorySaver()
app_rag = workflow.compile(checkpointer=checkpointer)
print("[INFO] Motor RAG conversacional V4 (Router + MMR + MemorySaver) compilado y listo en memoria.")


[INFO] Motor RAG conversacional V4 (Router + MMR + MemorySaver) compilado y listo en memoria.


---

## Suite de Evaluación Automatizada V4 y Verificación de Rutas (`Stress Testing`)

Para auditar de forma rigurosa el comportamiento del enrutador no lineal, la recuperación MMR y el checkpointer `MemorySaver`, ejecutamos una batería automatizada de **7 casos de control estratégicos (100% renovados respecto a v3)**:

* **Caso 1 (Consulta directa sobre RGPD - `PATH_LEGAL_RAG`):** Comprueba la recuperación e integración normativa del Artículo 35 sobre Evaluación de Impacto (EIPD).
* **Caso 2 (Razonamiento multi-turno con anáfora - `PATH_LEGAL_RAG`):** Verifica que el nodo reformulador reescribe correctamente una consulta dependiente (*"¿Y qué obligación legal vinculante tiene el responsable... al finalizar ese documento?"*) para recuperar la Consulta Previa obligatoria del Artículo 36 del RGPD.
* **Caso 3 (Consulta técnico-legal sobre IA Agéntica - `PATH_LEGAL_RAG`):** Evalúa la recuperación en las orientaciones sobre riesgos de privacidad al delegar acciones mediante conexiones externas o *tool calling*.
* **Caso 4 (Consulta multilingüe en inglés - Guardrail #5 Espejo Lingüístico - `PATH_LEGAL_RAG`):** Confirma la detección de idioma y redacción 100% en inglés al consultar sobre mecanismos de supervisión humana según el Dictamen 2024/28 del EDPB.
* **Caso 5 (Consulta Conversacional y de Conceptos Generales - `PATH_GENERAL_LLM`):** Pone a prueba la ruta directa de V4 al formular un saludo y solicitar una distinción conceptual básica entre LLM y agente autónomo, sin consumir cuota ni tiempo en la base vectorial.
* **Caso 6 (Guardrail de Dominio - `PATH_OUT_OF_DOMAIN`):** Intentamos desviar al agente preguntando cómo sustituir la correa de distribución y la bomba de agua de un motor diésel. El enrutador debe cortar en 0.1s con el mensaje exacto de rechazo.
* **Caso 7 (Guardrail de Ausencia de Información - `PATH_LEGAL_RAG` ajeno al corpus):** Consultamos las sanciones penales y de prisión exactas en el Código Penal de Japón para robótica industrial. Al ser materia jurídico-tecnológica no indexada, el modelo debe reconocer su límite sin alucinar.


In [8]:
# Ejecución automatizada de los 7 casos estratégicos renovados V4 con MemorySaver (`thread_id` aislado)
from IPython.display import display, Markdown
import time

consultas_evaluacion = [
    "¿En qué supuestos concretos es obligatorio que un Responsable del tratamiento realice una Evaluación de Impacto relativa a la Protección de Datos (EIPD) según el Artículo 35 del RGPD, y qué aspectos esenciales debe contener dicho documento?",
    "¿Y qué obligación legal vinculante tiene el responsable con la autoridad de control si, al finalizar ese documento que acabas de detallar, se advierte que el tratamiento entraña un alto riesgo pese a las medidas cautelares adoptadas?",
    "¿Qué riesgos de privacidad, minimización de datos y fugas de información advierten las orientaciones técnicas cuando un agente de IA autónomo se conecta dinámicamente con herramientas y APIs externas de terceros (tool calling o plugins) para ejecutar acciones?",
    "According to the EDPB Opinion 2024/28 and general data protection standards, what specific human oversight and transparency mechanisms are required when deploying an autonomous AI agent that takes automated decisions impacting individual rights?",
    "¡Hola, buenos días! ¿Podrías explicarme de manera breve y sencilla qué diferencia conceptual existe en informática entre un modelo de lenguaje masivo (LLM) y un agente autónomo de IA?",
    "¿Podrías explicarme paso a paso qué herramientas mecánicas necesito y cómo debo proceder para sustituir la correa de distribución y la bomba de agua en un motor diésel?",
    "¿Cuáles son las sanciones penales exactas y las penas de prisión previstas en el Código Penal de Japón para delitos de robo o filtración de secretos comerciales en empresas de robótica industrial?"
]

print("=" * 75)
print("[SUITE DE EVALUACION AUTOMATIZADA V4: ROUTER + MMR + MEMORYSAVER (7 CASOS RENOVALOS)]")
print("=" * 75 + "\n")

for idx, pregunta in enumerate(consultas_evaluacion, 1):
    print(f">>> PRUEBA DE ESFUERZO {idx} / {len(consultas_evaluacion)}")
    print(f"Consulta del Usuario: \"{pregunta}\"")
    
    t_inicio = time.time()
    # Aislamiento por hilo en MemorySaver para cada caso de prueba
    config_hilo = {"configurable": {"thread_id": f"hilo_evaluacion_caso_{idx}"}}
    estado_input = {"messages": [HumanMessage(content=pregunta)]}
    
    try:
        estado_final = app_rag.invoke(estado_input, config=config_hilo)
        t_duracion = time.time() - t_inicio
        
        intencion_detectada = estado_final.get("intencion", "LEGAL_RAG")
        print(f"[TRAZA V4] Intención clasificada por Enrutador: {intencion_detectada} | Latencia: {t_duracion:.2f}s")
        print("-" * 75)
        
        respuesta_generada = estado_final["messages"][-1].content
        if isinstance(respuesta_generada, list):
            respuesta_generada = "".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in respuesta_generada])
            
        display(Markdown(respuesta_generada))
    except Exception as exc:
        print(f"[ERROR] Fallo en la ejecucion del caso {idx}: {exc}")
    print("\n" + "=" * 75 + "\n")

print("[EXITO] Suite de evaluacion V4 completada. Todas las rutas condicionales han sido verificadas.")


[SUITE DE EVALUACION AUTOMATIZADA V4: ROUTER + MMR + MEMORYSAVER (7 CASOS RENOVALOS)]

>>> PRUEBA DE ESFUERZO 1 / 7
Consulta del Usuario: "¿En qué supuestos concretos es obligatorio que un Responsable del tratamiento realice una Evaluación de Impacto relativa a la Protección de Datos (EIPD) según el Artículo 35 del RGPD, y qué aspectos esenciales debe contener dicho documento?"
[TRAZA V4] Intención clasificada por Enrutador: LEGAL_RAG | Latencia: 2.95s
---------------------------------------------------------------------------


### Conclusión Directa
La realización de una Evaluación de Impacto relativa a la Protección de Datos (EIPD) es obligatoria cuando un tratamiento, especialmente mediante nuevas tecnologías, entrañe un alto riesgo para los derechos y libertades de las personas físicas.

### Análisis Normativo
El marco normativo establece los siguientes supuestos y consideraciones para la obligatoriedad de la EIPD:

*   **Criterio General de Riesgo:** Es preceptiva cuando el tratamiento, por su naturaleza, alcance, contexto o fines, suponga un alto riesgo para los derechos y libertades de los interesados.
*   **Supuestos Específicos:**
    *   Evaluación sistemática y exhaustiva de aspectos personales basada en tratamiento automatizado (incluyendo elaboración de perfiles) que derive en decisiones con efectos jurídicos o afectación significativa.
    *   Tratamiento a gran escala de categorías especiales de datos (Art. 9.1) o datos relativos a condenas e infracciones penales (Art. 10).
    *   Observación sistemática a gran escala de zonas de acceso público, incluyendo el uso de dispositivos optoelectrónicos.
    *   Operaciones que la autoridad de control considere que entrañan un alto riesgo, tales como aquellas que impidan el ejercicio de un derecho, el uso de un servicio o la ejecución de un contrato.
*   **Excepciones:** No se considera tratamiento a gran escala, y por tanto no requiere EIPD obligatoria, aquel realizado por un solo médico, profesional de la salud o abogado respecto a datos de sus pacientes o clientes.
*   **Procedimiento:** El responsable del tratamiento debe recabar el asesoramiento del Delegado de Protección de Datos (DPD), si ha sido nombrado, durante la realización de la evaluación. Una única evaluación puede cubrir operaciones similares que presenten riesgos análogos.

*Nota: La información disponible en la base de conocimientos no permite responder a la parte de su consulta relativa a los aspectos esenciales que debe contener el documento de la EIPD.*

### Trazabilidad Jurídica
* **reglamentoRGPD.md (RGPD -> General -> Artículo 35)** — Obligatoriedad de la EIPD, supuestos específicos, excepciones y rol del DPD.



>>> PRUEBA DE ESFUERZO 2 / 7
Consulta del Usuario: "¿Y qué obligación legal vinculante tiene el responsable con la autoridad de control si, al finalizar ese documento que acabas de detallar, se advierte que el tratamiento entraña un alto riesgo pese a las medidas cautelares adoptadas?"
[TRAZA V4] Intención clasificada por Enrutador: LEGAL_RAG | Latencia: 2.49s
---------------------------------------------------------------------------


### Conclusión Directa
El responsable tiene la obligación legal vinculante de consultar a la autoridad de control antes de iniciar el tratamiento de datos cuando la evaluación de impacto confirme la persistencia de un alto riesgo que no puede ser mitigado mediante medidas adecuadas.

### Análisis Normativo
*   **Obligación de consulta previa:** Cuando una evaluación de impacto relativa a la protección de datos (EIPD) determina que las operaciones de tratamiento entrañan un alto riesgo para los derechos y libertades de las personas físicas, y el responsable concluye que dicho riesgo no puede ser mitigado mediante medidas razonables (en términos de tecnología disponible y costes de aplicación), el responsable está obligado a consultar a la autoridad de control antes de proceder al tratamiento.
*   **Finalidad de la consulta:** Esta consulta tiene como objetivo asegurar el cumplimiento del Reglamento en escenarios donde el riesgo residual, tras la implementación de garantías y medidas de seguridad, sigue siendo elevado.
*   **Intervención de la autoridad:** Una vez presentada la solicitud, la autoridad de control tiene la facultad de asesorar por escrito al responsable si considera que el tratamiento previsto podría infringir la normativa, pudiendo ejercer los poderes de investigación y corrección que le son conferidos.
*   **Gestión de plazos:** La autoridad debe responder en un plazo de ocho semanas, prorrogable por seis semanas adicionales en casos de complejidad, debiendo informar al responsable sobre cualquier prórroga y los motivos que la justifican.

### Trazabilidad Jurídica
* **reglamentoRGPD.md (Reglamento General de Protección de Datos -> Sin Capitulo -> Sin Articulo)** — Obligación de consulta previa ante alto riesgo no mitigable, plazos de respuesta de la autoridad y facultades de asesoramiento.



>>> PRUEBA DE ESFUERZO 3 / 7
Consulta del Usuario: "¿Qué riesgos de privacidad, minimización de datos y fugas de información advierten las orientaciones técnicas cuando un agente de IA autónomo se conecta dinámicamente con herramientas y APIs externas de terceros (tool calling o plugins) para ejecutar acciones?"
[TRAZA V4] Intención clasificada por Enrutador: LEGAL_RAG | Latencia: 2.93s
---------------------------------------------------------------------------


...



>>> PRUEBA DE ESFUERZO 6 / 7
Consulta del Usuario: "¿Podrías explicarme paso a paso qué herramientas mecánicas necesito y cómo debo proceder para sustituir la correa de distribución y la bomba de agua en un motor diésel?"
[TRAZA V4] Intención clasificada por Enrutador: OUT_OF_DOMAIN | Latencia: 0.77s
---------------------------------------------------------------------------


No estoy entrenado para responder sobre ese tema



>>> PRUEBA DE ESFUERZO 7 / 7
Consulta del Usuario: "¿Cuáles son las sanciones penales exactas y las penas de prisión previstas en el Código Penal de Japón para delitos de robo o filtración de secretos comerciales en empresas de robótica industrial?"
[TRAZA V4] Intención clasificada por Enrutador: LEGAL_RAG | Latencia: 1.39s
---------------------------------------------------------------------------


La información disponible en la base de conocimientos no permite responder a esta consulta.



[EXITO] Suite de evaluacion V4 completada. Todas las rutas condicionales han sido verificadas.


---

## Celda Interactiva de Conversación en Vivo (Paso 6 del Enunciado)

Para dar cumplimiento al **Paso 6 del Enunciado Práctico**, incorporamos a continuación una celda ejecutable con un bucle conversacional interactivo. Esta celda permite al profesor, evaluador o estudiante realizar consultas libres en tiempo real directamente desde la interfaz del Jupyter Notebook.

### Funcionamiento del Bucle Interactivo:
* Al ejecutar la celda, el sistema inicializa un historial en blanco y solicita una pregunta por teclado mediante la función `input()`.
* La consulta atraviesa los tres nodos del grafo: reescritura de posibles referencias previas, búsqueda en la base vectorial de `ChromaDB` y redacción fundamentada por el modelo `Gemini Flash-Lite`.
* La respuesta del Consultor Legal se visualiza al momento renderizada en formato Markdown (`display(Markdown(...))`), organizada en sus tres encabezados jerárquicos sin numerar (`### Conclusión Directa`, `### Análisis Normativo` y `### Trazabilidad Jurídica`) y con sus fuentes desduplicadas.
* Para salir o terminar el chat en directo, basta con introducir en la celda las palabras `salir`, `fin` o `terminar`.


In [9]:
# Celda Interactiva de Conversacion en Vivo
# Ejecuta este bloque para interactuar y chatear libremente con el Consultor Legal RAG

print("=" * 68)
print("   MODO INTERACTIVO EN VIVO: CONSULTOR LEGAL RAG")
print("   Escribe tu consulta normativa o introduce 'salir' para terminar")
print("=" * 68 + "\n")

historial_interactivo = []

while True:
    try:
        pregunta_usuario = input("\nTú (Escribe tu pregunta al RAG): ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\n[INFO] Sesion interactiva finalizada por orden de interrupcion del usuario.")
        break
        
    if not pregunta_usuario:
        continue
        
    if pregunta_usuario.lower() in ["salir", "exit", "quit", "terminar", "fin"]:
        print("\n[INFO] Cerrando el chat interactivo. Gracias por utilizar el sistema RAG.")
        break
        
    print(f"\n> Consultando base vectorial y procesando grafo para: '{pregunta_usuario}'...")
    
    historial_interactivo.append(HumanMessage(content=pregunta_usuario))
    estado_input = {"messages": historial_interactivo}
    
    t_ini = time.time()
    try:
        config_interactivo = {"configurable": {"thread_id": "sesion_interactiva_envivo"}}
        estado_salida = app_rag.invoke(estado_input, config=config_interactivo)
        t_tot = time.time() - t_ini
        
        historial_interactivo = estado_salida["messages"]
        respuesta_agente = historial_interactivo[-1].content
        
        print(f"\n--- Respuesta del Consultor Legal RAG ({t_tot:.2f}s) ---")
        texto_md_agente = respuesta_agente if isinstance(respuesta_agente, str) else "".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in respuesta_agente])
        display(Markdown(texto_md_agente))
        print("-" * 68)
    except Exception as exc:
        print(f"\n[ADVERTENCIA] Ha ocurrido un error temporal de red o cuota al consultar la API: {exc}")
        print("Sugerencia: Si la API de Google esta saturada en este instante (Error 429/503), espera unos segundos y vuelve a preguntar.")


   MODO INTERACTIVO EN VIVO: CONSULTOR LEGAL RAG
   Escribe tu consulta normativa o introduce 'salir' para terminar


> Consultando base vectorial y procesando grafo para: '¿Cuáles son los requisitos normativos exactos del derecho a la limitación del tratamiento regulado en el Artículo 18 del RGPD y en qué supuestos específicos puede el interesado solicitar que se congele el tratamiento de sus datos?'...

--- Respuesta del Consultor Legal RAG (3.18s) ---


### Conclusión Directa
El derecho a la limitación del tratamiento permite al interesado restringir el uso de sus datos personales cuando concurren situaciones específicas de controversia, ilicitud, necesidad de defensa legal o verificación de motivos legítimos.

### Análisis Normativo
El ejercicio del derecho a la limitación del tratamiento, conforme a la normativa, se articula bajo los siguientes preceptos:

*   **Supuestos habilitantes:** El interesado puede solicitar la limitación del tratamiento en cuatro escenarios taxativos:
    1.  **Impugnación de exactitud:** Cuando el interesado cuestiona la veracidad de los datos, durante el tiempo necesario para que el responsable verifique dicha exactitud.
    2.  **Tratamiento ilícito:** Cuando el tratamiento es contrario a la norma y el interesado prefiere la limitación a la supresión de los datos.
    3.  **Defensa de reclamaciones:** Cuando el responsable ya no requiere los datos para los fines originales, pero el interesado los necesita para formular, ejercer o defender reclamaciones.
    4.  **Oposición pendiente:** Cuando el interesado se ha opuesto al tratamiento (conforme al art. 21.1 RGPD) mientras se verifica si los motivos legítimos del responsable prevalecen sobre los del interesado.
*   **Efectos del tratamiento limitado:** Una vez limitada la restricción, los datos solo podrán ser objeto de tratamiento (salvo su conservación) con el consentimiento del interesado, para la defensa de reclamaciones, para la protección de derechos de terceros o por razones de interés público importante.
*   **Obligaciones del responsable:** El responsable debe informar al interesado antes de proceder al levantamiento de la limitación.
*   **Implementación técnica:** En entornos automatizados, la limitación debe ejecutarse mediante medios técnicos que impidan operaciones de tratamiento ulterior o modificaciones, debiendo indicarse claramente en el sistema que los datos están limitados. Los métodos pueden incluir el traslado temporal a otro sistema, la restricción de acceso o la retirada temporal de publicaciones en internet.

### Trazabilidad Jurídica
* **reglamentoRGPD.md (RGPD -> General -> Sin Articulo)** — Regulación de los supuestos de limitación, las condiciones de tratamiento durante la limitación, la obligación de informar al interesado y las especificaciones técnicas para la ejecución de la limitación (Considerando 67).

--------------------------------------------------------------------

> Consultando base vectorial y procesando grafo para: '¿Y bajo qué condiciones estrictas o excepciones legales permite la norma al responsable seguir tratando esos datos cuya congelación o limitación acaba de ser ejercitada en el paso anterior?'...

--- Respuesta del Consultor Legal RAG (3.40s) ---


### Conclusión Directa
El responsable del tratamiento solo puede procesar datos personales sujetos a limitación si cuenta con el consentimiento del interesado, si el tratamiento es necesario para la defensa de reclamaciones, para proteger los derechos de terceros, o por razones de interés público importante.

### Análisis Normativo
Una vez que el derecho a la limitación del tratamiento ha sido ejercido, el responsable debe cesar cualquier operación de tratamiento, salvo la mera conservación, a menos que concurra alguna de las siguientes excepciones taxativas:

*   **Consentimiento explícito:** El tratamiento es lícito si el interesado otorga su consentimiento para operaciones específicas sobre dichos datos.
*   **Defensa de derechos:** Se permite el tratamiento cuando sea necesario para la formulación, el ejercicio o la defensa de reclamaciones legales.
*   **Protección de terceros:** El tratamiento es admisible cuando tenga como finalidad la protección de los derechos de otra persona física o jurídica.
*   **Interés público:** Se habilita el tratamiento por razones de interés público importante, ya sea a nivel de la Unión Europea o de un Estado miembro determinado.
*   **Obligación de información:** El responsable del tratamiento tiene la obligación imperativa de informar al interesado antes de proceder al levantamiento de la limitación del tratamiento.

### Trazabilidad Jurídica
* **reglamentoRGPD.md (RGPD -> General -> Sin Articulo)** — Regulación de las excepciones al tratamiento de datos limitados (consentimiento, defensa de reclamaciones, protección de derechos de terceros e interés público) y la obligación de informar al interesado antes del levantamiento de la limitación.

--------------------------------------------------------------------

> Consultando base vectorial y procesando grafo para: '¿Qué directrices normativas e implicaciones de seguridad establece la guía respecto a la memoria a largo plazo (long-term memory / bases vectoriales) en agentes de IA, y cómo debe gestionarse el principio de exactitud y rectificación de datos almacenados en dichas memorias?'...

--- Respuesta del Consultor Legal RAG (4.04s) ---


...

--------------------------------------------------------------------

> Consultando base vectorial y procesando grafo para: '¿Qué requisitos técnicos y de ciberseguridad exige la legislación bancaria federal de Suiza respecto a los centros de datos en la nube para entidades financieras locales?'...

--- Respuesta del Consultor Legal RAG (2.19s) ---


La información disponible en la base de conocimientos no permite responder a esta consulta.

--------------------------------------------------------------------

> Consultando base vectorial y procesando grafo para: 'Que es lo mas importante del rgpd'...

--- Respuesta del Consultor Legal RAG (3.42s) ---


### Conclusión Directa
El Reglamento General de Protección de Datos (RGPD) tiene como objeto principal garantizar la protección de las personas físicas en el tratamiento de sus datos personales y asegurar la libre circulación de dicha información dentro de la Unión Europea.

### Análisis Normativo
* **Protección de Derechos Fundamentales:** El reglamento salvaguarda los derechos y libertades fundamentales de las personas físicas, con especial énfasis en el derecho a la protección de sus datos personales.
* **Ámbito de Aplicación:** Establece normas detalladas que abarcan desde la recogida y seudonimización de datos hasta la gestión de la seguridad del tratamiento y la notificación de violaciones de seguridad a las autoridades de control.
* **Derechos de los Interesados:** Regula el ejercicio de los derechos de los interesados, incluyendo el derecho a la supresión de datos y la portabilidad, permitiendo que los datos sean transmitidos directamente entre responsables cuando sea técnicamente posible.
* **Transparencia y Flujo de Datos:** Se enfatiza la importancia de la información proporcionada al público y a los interesados, así como la necesidad de garantizar la transparencia en el flujo de datos, incluso superando los mínimos legales para demostrar confianza en entornos de IA agéntica.
* **Protección Específica:** Incluye disposiciones sobre la protección de menores y la obtención de consentimiento por parte de los titulares de la patria potestad o tutela.
* **Transferencias Internacionales:** Regula las condiciones bajo las cuales se pueden realizar transferencias de datos personales a terceros países u organizaciones internacionales.

### Trazabilidad Jurídica
* **reglamentoRGPD.md (General -> Sin Capitulo -> Sin Articulo)** — Objeto del reglamento, protección de derechos fundamentales y libre circulación de datos.
* **reglamentoRGPD.md (General -> Sin Capitulo -> Sin Articulo)** — Alcance operativo sobre recogida, seudonimización, derechos de los interesados, seguridad, notificaciones y transferencias internacionales.
* **reglamentoRGPD.md (General -> Sin Capitulo -> Sin Articulo)** — Disposiciones sobre supresión de datos, ejecución de contratos y portabilidad.
* **Orientaciones Ia Agéntica.md (General -> Sin Capitulo -> Sin Articulo)** — Requisitos de transparencia y flujo de datos.

--------------------------------------------------------------------

[INFO] Cerrando el chat interactivo. Gracias por utilizar el sistema RAG.
